# AAIT (III–II) – EXP-16
## Building an AI What-If Scenario Simulator (Interactive Runtime Version)

**Aim:** To implement the core components of an end-to-end AI solution, loading a dataset dynamically at runtime, building a regression model, and simulating runtime interactions via console inputs.
- **Model:** Linear Regression
- **Domain:** Student Performance (Marks Prediction)

**Tools:** Python (VS Code / Google Colab), pandas, scikit-learn, matplotlib
**Mapped CO:** CO5

---
### What you will build (End result)
An AI **What-If Simulator** that predicts student marks by asking for real-time user inputs during execution:
- Student Name
- Study Hours
- Attendance (%)
- Sleep Hours


## Step 1 & 2 — Define Problem and Prepare Dataset at Runtime

Instead of hardcoding the data, we will load an external `data.csv` dataset directly during runtime.

In [ ]:
# EXP-16: AI What-If Simulator (Interactive)
# -------------------------------------------------
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import os

print("--- Step 2: Preparing Dataset (Runtime) ---")

# Use the local data/data.csv file if it exists, otherwise ask the user to input the path
file_path = "data/data.csv"
if not os.path.exists(file_path):
    file_path = input("File not found automatically. Enter the path to your dataset CSV: ").strip()

try:
    data = pd.read_csv(file_path)
    print(f"\nDataset loaded successfully from '{file_path}' with {len(data)} student values.\n")
    display(data.head())
except Exception as e:
    print(f"Error loading dataset: {e}")

## Step 3 & 4 — Choose and Train the Model

Since the output (marks) is a continuous number, we use **Linear Regression**.
First, we isolate the numeric columns (dropping student_name) and split the runtime data into training (70%) and testing (30%) sets.

In [ ]:
# Features (X) and Target (y) - We drop 'student_name' since Regression only takes numbers
X = data[['study_hours', 'attendance', 'sleep']]
y = data['marks']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Data split: {len(X_train)} training rows, {len(X_test)} testing rows.\n")

print("--- Training the Model ---")
model = LinearRegression()
model.fit(X_train, y_train)
print("Linear Regression model trained successfully.")

## Step 5a — Evaluate the Result (Metrics)

We predict on the unseen testing set and measure the errors (MSE, MAE) and variance explained (R2 Score).

In [ ]:
print("--- Step 5: Evaluating the Result ---")
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R-squared (R2 Score):     {r2:.2f}")

## Step 5b — Visualize the Results

Let's plot the **Actual vs Predicted Marks** to check our AI's accuracy visually.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, color='blue', s=80, alpha=0.3, label='Model Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Fit')

plt.xlabel("Actual Marks (Ground Truth)", fontsize=11)
plt.ylabel("Predicted Marks (AI Simulation)", fontsize=11)
plt.title("Actual vs Predicted Marks Evaluation", fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## Step 6 — Interactive Runtime UI Loop

This cell recreates the chatbot-style interaction loop from Experiment 12. 
It continuously asks you for runtime input (Student Name, Study Hours, Attendance, Sleep) and instantly provides the prediction until you type `exit`.

In [ ]:
def run_interactive_simulator():
    print("\n=========================================")
    print("     AI WHAT-IF RUNTIME SIMULATOR        ")
    print("     (Extended MVP Decision System)      ")
    print("=========================================\n")
    
    while True:
        print("\nOPTIONS:")
        print("1. Scenario Simulation (Multiple Inputs)")
        print("2. Predict & Get Recommendations")
        print("3. Goal-Based Planning (Set Target Marks)")
        print("Type 'exit' to stop.")
        
        choice = input("\nSelect an option (1/2/3/exit): ").strip().lower()
        if choice == 'exit':
            break
            
        if choice == '1':
            print("\n--- 1. SCENARIO SIMULATION ---")
            print("We analyze different scenarios to understand the impact of study hours.")
            name = input("Enter Student Name: ").strip()
            try:
                att = float(input("Baseline Attendance % (40-100): "))
                slp = float(input("Baseline Sleep Hours (4-12): "))
            except ValueError:
                print("[Error] Please enter valid numbers.")
                continue
            
            print("\n  ▶ MULTIPLE SITUATIONS FOR", name.upper())
            for study_hrs in [2, 6, 9]:
                scenario = pd.DataFrame([[study_hrs, att, slp]], columns=['study_hours', 'attendance', 'sleep'])
                mark = model.predict(scenario)[0]
                qualifier = "low" if study_hrs <= 3 else "medium" if study_hrs <= 6 else "high"
                print(f"    If study = {study_hrs} hrs -> predicted marks = {mark:.1f} ({qualifier} performance)")
            print("-" * 40)
            
        elif choice == '2':
            print("\n--- 2. PREDICT & RECOMMEND ---")
            name = input("Enter Student Name: ").strip()
            try:
                std = float(input("Study Hours (0-10): "))
                att = float(input("Attendance % (40-100): "))
                slp = float(input("Sleep Hours (4-12): "))
            except ValueError:
                print("[Error] Please enter valid numbers.")
                continue
            
            scenario = pd.DataFrame([[std, att, slp]], columns=['study_hours', 'attendance', 'sleep'])
            pred_marks = model.predict(scenario)[0]
            
            print("\n  ▶ SYSTEM OUTPUT & RECOMMENDATIONS:")
            print(f"    {name}, your Predicted Marks = {pred_marks:.1f}")
            print("\n    [!] Required Improvements:")
            improved = False
            if std < 5:
                print("    - Increase study hours")
                improved = True
            if att < 75:
                print("    - Improve attendance")
                improved = True
            if slp < 6 or slp > 9:
                print("    - Maintain proper sleep")
                improved = True
                
            if not improved:
                print("    - Great job! Your habits are perfectly aligned for success.")
            print("-" * 40)
            
        elif choice == '3':
            print("\n--- 3. GOAL-BASED PLANNING ---")
            name = input("Enter Student Name: ").strip()
            target_str = input(f"What is your Target Marks, {name}? (0-100): ").strip()
            
            try:
                target = float(target_str)
                coeff_study = model.coef_[0]
                coeff_att = model.coef_[1]
                coeff_sleep = model.coef_[2]
                intercept = model.intercept_
                
                fixed_att = 85.0
                fixed_slp = 7.0
                
                required_study = (target - intercept - (coeff_att * fixed_att) - (coeff_sleep * fixed_slp)) / coeff_study
                req_study_clamped = max(0.0, required_study)
                
                print("\n  ▶ GOAL ROADMAP & OPTIMIZATION:")
                print(f"    To achieve a Target of {target} marks, the system suggests:")
                print(f"    → Attendance = {fixed_att}%")
                print(f"    → Sleep = {fixed_slp} hrs")
                print(f"    → Requirement: Study = {req_study_clamped:.1f} hrs")
                print("-" * 40)
                
            except ValueError:
                print("\n[Error] Invalid target value.")
        else:
            print("Invalid choice. Try again.")
            
    print("\nSimulator closed. Goodbye!")

run_interactive_simulator()